# Role 3: real CLIP-to-temporal validation

This notebook validates the committed Role 3 pipeline on a small number of actual ReVA **validation** videos. It checks tensor shapes, finite outputs, frozen CLIP behavior, and temporal gradients. It does not score answers or train a model.

Before running: choose **Runtime → Change runtime type → T4 GPU** if available. Colab Free does not guarantee a GPU. The notebook clones `suraj-role3-foundation` and `sarthak` into separate folders; it does not edit or merge either branch. It loads the pinned `val.json` annotation file and fetches only the selected video's file through Sarthak's lazy Hub loader.

The notebook writes measurements (or explicit failures/nulls) to `reports/role3_real_inference_results.json` in the cloned Role 3 tree. No performance numbers are assumed.

In [ ]:
%pip install -q "transformers>=4.40,<5" "huggingface_hub>=0.23,<1" opencv-python-headless

# Colab supplies PyTorch and NumPy. They are intentionally not reinstalled.

In [ ]:
import torch
import platform

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = None
GPU_TOTAL_MEMORY_MB = None
GPU_FREE_MEMORY_MB = None

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    GPU_TOTAL_MEMORY_MB = round(total_bytes / 1024**2, 1)
    GPU_FREE_MEMORY_MB = round(free_bytes / 1024**2, 1)
    print("GPU:", GPU_NAME)
    print(f"GPU memory: {GPU_FREE_MEMORY_MB:.1f} MiB free / {GPU_TOTAL_MEMORY_MB:.1f} MiB total")
else:
    print("No CUDA GPU assigned; the notebook will attempt the small experiment on CPU.")

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/sarthak-shastri/ChronoMind.git"
ROLE3_BRANCH = "suraj-role3-foundation"
PREPROCESSING_BRANCH = "sarthak"
ROLE3_ROOT = Path("/content/ChronoMind-role3")
SARTHAK_ROOT = Path("/content/ChronoMind-sarthak")


def clone_branch(branch, destination):
    if not destination.exists():
        subprocess.run(
            ["git", "clone", "--quiet", "--single-branch", "--branch", branch,
             REPO_URL, str(destination)],
            check=True,
            text=True,
            capture_output=True,
        )
    else:
        current = subprocess.check_output(
            ["git", "-C", str(destination), "branch", "--show-current"], text=True
        ).strip()
        dirty = subprocess.check_output(
            ["git", "-C", str(destination), "status", "--porcelain"], text=True
        ).strip()
        if current != branch:
            raise RuntimeError(
                f"{destination} already exists on branch {current!r}; expected {branch!r}. "
                "Use a fresh Colab runtime or remove that runtime folder manually."
            )
        if dirty:
            raise RuntimeError(
                f"{destination} contains local changes. Use a fresh Colab runtime; "
                "the notebook will not overwrite them."
            )


clone_branch(ROLE3_BRANCH, ROLE3_ROOT)
clone_branch(PREPROCESSING_BRANCH, SARTHAK_ROOT)

role3_commit = subprocess.check_output(
    ["git", "-C", str(ROLE3_ROOT), "rev-parse", "HEAD"], text=True
).strip()
sarthak_commit = subprocess.check_output(
    ["git", "-C", str(SARTHAK_ROOT), "rev-parse", "HEAD"], text=True
).strip()

required_role3_files = [
    ROLE3_ROOT / "advanced_models/temporal/temporal_encoder.py",
    ROLE3_ROOT / "advanced_models/vision/feature_extractor.py",
    ROLE3_ROOT / "advanced_models/video_temporal_pipeline.py",
]
missing_role3_files = [str(path) for path in required_role3_files if not path.is_file()]
if missing_role3_files:
    raise FileNotFoundError(
        "Role 3 implementation is missing from the cloned branch: "
        + ", ".join(missing_role3_files)
        + ". Push the Role 3 implementation to suraj-role3-foundation, then rerun."
    )

print("Role 3 branch/commit:", ROLE3_BRANCH, role3_commit)
print("Preprocessing branch/commit:", PREPROCESSING_BRANCH, sarthak_commit)

In [ ]:
import sys

# Keep both checkouts separate. Role 3 modules resolve from ROLE3_ROOT; the
# top-level preprocessing package resolves from SARTHAK_ROOT.
sys.path.insert(0, str(SARTHAK_ROOT))
sys.path.insert(0, str(ROLE3_ROOT))

import cv2
import numpy as np
import transformers
from preprocessing import ReVADataset, make_dataloader, read_annotations
from preprocessing.hf_access import fetch_annotation_file
from advanced_models.temporal.temporal_encoder import TransformerTemporalEncoder
from advanced_models.video_temporal_pipeline import VideoTemporalPipeline
from advanced_models.vision.feature_extractor import CLIP_MODEL_ID, CLIPVisionFeatureExtractor
from transformers import CLIPVisionModelWithProjection

print("Transformers:", transformers.__version__)
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Preprocessing imported from:", Path(sys.modules["preprocessing"].__file__).resolve())
print("Role 3 pipeline imported from:", Path(sys.modules["advanced_models.video_temporal_pipeline"].__file__).resolve())

## Load the actual validation annotation and a small sample

Sarthak's committed manifest pins dataset revision `06224fa1b42e5e35677a8b7f2145e06307a9a4ee` and lists `val.json`. The branch's inspection report records 2,000 validation records and 379 unique video paths. The code below verifies that `val.json` is present in that manifest and selects a sample ID from the downloaded annotation itself; it does not hardcode a QA record or video path.

In [ ]:
import json

DATASET_REPO_ID = "ReVA-Benchmark/ReVA"
FRAME_COUNTS = (4, 8, 16)
SAMPLE_LIMIT = 1
CACHE_ROOT = Path("/content/reva_role3_cache")
HF_CACHE_DIR = CACHE_ROOT / "huggingface"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = SARTHAK_ROOT / "manifests/reva_manifest.json"
DATA_REVISION = None
ANNOTATION_FILENAME = None
ANNOTATION_PATH = None
VALIDATION_ROWS = []
SAMPLE_IDS = []
ACTUAL_VIDEO_PATHS = []
DATA_SETUP_ERROR = None
DATA_ACCESS_ERRORS = {}
VALIDATION_BATCHES = {}

try:
    with manifest_path.open(encoding="utf-8") as handle:
        manifest = json.load(handle)
    DATA_REVISION = manifest["revision"]
    if "val.json" not in manifest["files"]:
        candidates = [name for name in manifest["files"] if "val" in Path(name).name.lower() and name.endswith(".json")]
        raise FileNotFoundError(
            f"Pinned manifest does not contain val.json. Validation-like files: {candidates}"
        )
    ANNOTATION_FILENAME = "val.json"
    ANNOTATION_PATH = fetch_annotation_file(
        DATASET_REPO_ID,
        ANNOTATION_FILENAME,
        revision=DATA_REVISION,
        cache_dir=HF_CACHE_DIR,
    )
    VALIDATION_ROWS = read_annotations(ANNOTATION_PATH)
    if not VALIDATION_ROWS:
        raise ValueError(f"No records found in {ANNOTATION_PATH}")

    def actual_sample_id(row):
        for key in ("qa_id", "sample_id", "id"):
            if row.get(key) is not None:
                return str(row[key])
        raise KeyError("Validation record has no qa_id/sample_id/id")

    selected_rows = VALIDATION_ROWS[:SAMPLE_LIMIT]
    SAMPLE_IDS = [actual_sample_id(row) for row in selected_rows]
    ACTUAL_VIDEO_PATHS = [row.get("video_path", row.get("file_path")) for row in selected_rows]
    print("Dataset revision:", DATA_REVISION)
    print("Actual annotation path:", ANNOTATION_PATH)
    print("Validation records:", len(VALIDATION_ROWS))
    print("Selected sample IDs:", SAMPLE_IDS)
    print("Selected video paths from val.json:", ACTUAL_VIDEO_PATHS)
except Exception as exc:
    DATA_SETUP_ERROR = {"type": type(exc).__name__, "message": str(exc)}
    print(f"DATA ACCESS ERROR ({type(exc).__name__}): {exc}")
    print("Resolution: verify Hugging Face access, the pinned dataset revision, and the exact val.json filename in the Sarthak manifest. If the repository requires authentication, authenticate in Colab and rerun this cell.")

In [ ]:
def make_validation_batch(num_frames):
    dataset = ReVADataset(
        annotation_file=ANNOTATION_PATH,
        video_root=None,
        num_frames=num_frames,
        sampling="uniform",
        split="val",
        sample_ids=SAMPLE_IDS,
        image_size=(224, 224),
        normalize=True,
        cache_dir=None,
        missing_video="raise",
        dataset_repo_id=DATASET_REPO_ID,
        revision=DATA_REVISION,
        hf_cache_dir=HF_CACHE_DIR,
        local_files_only=False,
    )
    if len(dataset) != len(SAMPLE_IDS):
        raise ValueError(
            f"Dataset produced {len(dataset)} selected rows; expected {len(SAMPLE_IDS)}"
        )
    loader = make_dataloader(
        dataset, batch_size=1, shuffle=False, num_workers=0
    )
    return dataset, next(iter(loader))

if DATA_SETUP_ERROR is None:
    for frame_count in FRAME_COUNTS:
        try:
            dataset, batch = make_validation_batch(frame_count)
            frames = batch.get("video_frames")
            if not torch.is_tensor(frames):
                raise TypeError("collate_reva did not return tensor video_frames")
            if tuple(frames.shape) != (1, frame_count, 3, 224, 224):
                raise ValueError(f"Unexpected video_frames shape: {tuple(frames.shape)}")
            if frames.dtype != torch.float32:
                raise TypeError(f"Expected float32 frames, got {frames.dtype}")
            if not torch.isfinite(frames).all():
                raise ValueError("video_frames contains NaN or Inf")

            # Verify that the actual source code performs BGR -> RGB conversion.
            loader_source = (SARTHAK_ROOT / "preprocessing/video_loader.py").read_text(encoding="utf-8")
            assert "COLOR_BGR2RGB" in loader_source, "Sarthak decoder no longer confirms RGB conversion"

            # Reverse the documented ImageNet normalization and ensure recovered
            # pixel values remain in the RGB [0,1] range (allowing tiny float error).
            mean = frames.new_tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)
            std = frames.new_tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)
            rgb01 = frames * std + mean
            if rgb01.min().item() < -1e-4 or rgb01.max().item() > 1.0001:
                raise ValueError("ImageNet-normalized frame values did not recover to RGB [0,1]")

            VALIDATION_BATCHES[frame_count] = batch
            print(f"T={frame_count}: shape={tuple(frames.shape)}, dtype={frames.dtype}, RGB conversion confirmed, ImageNet normalization verified")
        except Exception as exc:
            DATA_ACCESS_ERRORS[str(frame_count)] = {
                "type": type(exc).__name__, "message": str(exc)
            }
            print(f"DATA ACCESS ERROR for T={frame_count} ({type(exc).__name__}): {exc}")
            print("Resolution: check access to this actual video_path in the pinned ReVA revision, Hub connectivity, and OpenCV decoding.")
else:
    print("Skipping video loading because validation annotation setup failed.")

## Load the pretrained visual tower and compose the Role 3 pipeline

The model load is explicit and uses the required Hugging Face class and checkpoint. Only the vision tower is loaded. The temporal encoder is created with 512 input dimensions and supports the requested maximum sequence length.

In [ ]:
MODEL_LOAD_ERROR = None
MODEL_REVISION = None
PIPELINE = None
VISION_MODEL = None
TEMPORAL_ENCODER = None

try:
    VISION_MODEL = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL_ID)
    MODEL_REVISION = getattr(VISION_MODEL.config, "_commit_hash", None)
    extractor = CLIPVisionFeatureExtractor(
        VISION_MODEL,
        input_normalization="imagenet",
        frame_chunk_size=4,
        cuda_mixed_precision=torch.cuda.is_available(),
        output_dtype=torch.float32,
    )
    TEMPORAL_ENCODER = TransformerTemporalEncoder(
        embedding_dim=512,
        num_heads=8,
        num_layers=2,
        dropout=0.1,
        max_seq_len=16,
    )
    PIPELINE = VideoTemporalPipeline(extractor, TEMPORAL_ENCODER).to(DEVICE)
    PIPELINE.eval()
    print("Checkpoint:", CLIP_MODEL_ID)
    print("Checkpoint revision:", MODEL_REVISION)
    print("Pipeline device:", DEVICE)
except Exception as exc:
    MODEL_LOAD_ERROR = {"type": type(exc).__name__, "message": str(exc)}
    print(f"MODEL LOAD ERROR ({type(exc).__name__}): {exc}")
    print("Resolution: verify Colab internet access and available disk/RAM/GPU memory, then rerun this cell. No alternative checkpoint is substituted.")

## Run 4-, 8-, and 16-frame inference

The reported time covers pipeline forward after the input batch has been moved to the selected device; it excludes dataset download, video decode, and host-to-device transfer. CUDA peak allocated memory is measured after model loading and input transfer. If CUDA runs out of memory, the notebook halves the frame chunk size and retries down to one frame.

In [ ]:
import time

RUN_RESULTS = []


def run_pipeline_measurement(frame_count, batch):
    if PIPELINE is None:
        return {
            "frame_count": frame_count,
            "status": "failure",
            "error": MODEL_LOAD_ERROR or {"type": "Unavailable", "message": "Pipeline was not loaded"},
            "input_shape": None,
            "frame_features_shape": None,
            "video_features_shape": None,
            "inference_time_seconds": None,
            "peak_gpu_memory_mb": None,
            "frame_chunk_size": None,
        }

    input_cpu = batch["video_frames"]
    inputs = input_cpu.to(DEVICE)
    while True:
        chunk_size = PIPELINE.feature_extractor.frame_chunk_size
        try:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats()
            start = time.perf_counter()
            with torch.inference_mode():
                outputs = PIPELINE({"video_frames": inputs})
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
            frame_features = outputs["frame_features"]
            video_features = outputs["video_features"]
            peak_memory = (
                round(torch.cuda.max_memory_allocated() / 1024**2, 2)
                if torch.cuda.is_available()
                else None
            )
            result = {
                "frame_count": frame_count,
                "status": "success",
                "error": None,
                "input_shape": list(inputs.shape),
                "frame_features_shape": list(frame_features.shape),
                "video_features_shape": list(video_features.shape),
                "output_dtype": str(video_features.dtype),
                "output_device": str(video_features.device),
                "contains_nan": bool(torch.isnan(frame_features).any().item() or torch.isnan(video_features).any().item()),
                "contains_inf": bool(torch.isinf(frame_features).any().item() or torch.isinf(video_features).any().item()),
                "inference_time_seconds": elapsed,
                "peak_gpu_memory_mb": peak_memory,
                "frame_chunk_size": chunk_size,
            }
            print(
                f"T={frame_count}: input={result['input_shape']}, "
                f"frame_features={result['frame_features_shape']}, "
                f"video_features={result['video_features_shape']}, "
                f"dtype={result['output_dtype']}, device={result['output_device']}, "
                f"NaN={result['contains_nan']}, Inf={result['contains_inf']}, "
                f"time={elapsed:.4f}s, peak_GPU_MiB={peak_memory}, status=success"
            )
            return result
        except torch.cuda.OutOfMemoryError as exc:
            if not torch.cuda.is_available() or chunk_size is None or chunk_size <= 1:
                return {
                    "frame_count": frame_count,
                    "status": "failure",
                    "error": {"type": type(exc).__name__, "message": str(exc)},
                    "input_shape": list(inputs.shape),
                    "frame_features_shape": None,
                    "video_features_shape": None,
                    "inference_time_seconds": None,
                    "peak_gpu_memory_mb": round(torch.cuda.max_memory_allocated() / 1024**2, 2),
                    "frame_chunk_size": chunk_size,
                }
            next_chunk_size = max(1, chunk_size // 2)
            print(f"CUDA OOM at chunk size {chunk_size}; retrying with {next_chunk_size}.")
            PIPELINE.feature_extractor.frame_chunk_size = next_chunk_size
            torch.cuda.empty_cache()
        except Exception as exc:
            return {
                "frame_count": frame_count,
                "status": "failure",
                "error": {"type": type(exc).__name__, "message": str(exc)},
                "input_shape": list(inputs.shape),
                "frame_features_shape": None,
                "video_features_shape": None,
                "inference_time_seconds": None,
                "peak_gpu_memory_mb": None,
                "frame_chunk_size": chunk_size,
            }


for frame_count in FRAME_COUNTS:
    if str(frame_count) in DATA_ACCESS_ERRORS:
        RUN_RESULTS.append({
            "frame_count": frame_count,
            "status": "failure",
            "error": DATA_ACCESS_ERRORS[str(frame_count)],
            "input_shape": None,
            "frame_features_shape": None,
            "video_features_shape": None,
            "inference_time_seconds": None,
            "peak_gpu_memory_mb": None,
            "frame_chunk_size": None,
        })
    elif frame_count not in VALIDATION_BATCHES:
        RUN_RESULTS.append({
            "frame_count": frame_count,
            "status": "not_executed",
            "error": DATA_SETUP_ERROR,
            "input_shape": None,
            "frame_features_shape": None,
            "video_features_shape": None,
            "inference_time_seconds": None,
            "peak_gpu_memory_mb": None,
            "frame_chunk_size": None,
        })
    else:
        RUN_RESULTS.append(run_pipeline_measurement(frame_count, VALIDATION_BATCHES[frame_count]))

In [ ]:
from IPython.display import Markdown, display

headers = ["Frames", "Frame feature dims", "Video feature dims", "Inference seconds", "Peak GPU MiB", "Status"]
rows = []
for result in RUN_RESULTS:
    show = lambda value: "—" if value is None else str(value)
    rows.append([
        result["frame_count"],
        show(result.get("frame_features_shape")),
        show(result.get("video_features_shape")),
        show(round(result["inference_time_seconds"], 4) if result.get("inference_time_seconds") is not None else None),
        show(result.get("peak_gpu_memory_mb")),
        result["status"],
    ])
lines = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
lines.extend("| " + " | ".join(map(str, row)) + " |" for row in rows)
display(Markdown("\n".join(lines)))

## Verify frozen CLIP and temporal gradients

This performs one small backward pass through the temporal encoder only. It is not a training loop. The pipeline is deliberately called without an outer `no_grad`/`inference_mode` context so temporal gradients can be checked.

In [ ]:
TRAINING_CHECK = {"status": "not_executed"}

if PIPELINE is None:
    TRAINING_CHECK = {"status": "not_executed", "reason": "Pretrained pipeline failed to load"}
elif 4 not in VALIDATION_BATCHES:
    TRAINING_CHECK = {"status": "not_executed", "reason": "No actual 4-frame validation batch was loaded"}
else:
    try:
        PIPELINE.train()
        visual_frozen = all(not parameter.requires_grad for parameter in PIPELINE.feature_extractor.vision_model.parameters())
        visual_eval = not PIPELINE.feature_extractor.vision_model.training
        temporal_train = PIPELINE.temporal_encoder.training

        PIPELINE.zero_grad(set_to_none=True)
        train_inputs = VALIDATION_BATCHES[4]["video_frames"].to(DEVICE)
        train_outputs = PIPELINE({"video_frames": train_inputs})
        train_outputs["video_features"].square().mean().backward()
        temporal_has_grad = any(
            parameter.grad is not None
            for parameter in PIPELINE.temporal_encoder.parameters()
            if parameter.requires_grad
        )
        visual_has_grad = any(
            parameter.grad is not None
            for parameter in PIPELINE.feature_extractor.vision_model.parameters()
        )
        PIPELINE.eval()
        temporal_eval = not PIPELINE.temporal_encoder.training
        visual_still_eval = not PIPELINE.feature_extractor.vision_model.training

        passed = all([visual_frozen, visual_eval, temporal_train, temporal_has_grad, not visual_has_grad, temporal_eval, visual_still_eval])
        TRAINING_CHECK = {
            "status": "success" if passed else "failure",
            "visual_parameters_frozen": visual_frozen,
            "visual_model_eval_during_training": visual_eval,
            "temporal_encoder_train_mode": temporal_train,
            "temporal_gradients_present": temporal_has_grad,
            "visual_gradients_absent": not visual_has_grad,
            "temporal_eval_mode_after_pipeline_eval": temporal_eval,
            "visual_model_still_eval": visual_still_eval,
        }
        print(TRAINING_CHECK)
    except Exception as exc:
        TRAINING_CHECK = {"status": "failure", "error": {"type": type(exc).__name__, "message": str(exc)}}
        print(f"TRAINING BEHAVIOR CHECK ERROR ({type(exc).__name__}): {exc}")
    finally:
        PIPELINE.eval()
        PIPELINE.zero_grad(set_to_none=True)

## Save a reproducibility report

The report stores only values measured by the preceding cells. If data/model access or inference failed, the error text is recorded and unavailable measurements remain `null`. The output is saved inside the cloned Role 3 repository; set `DOWNLOAD_RESULTS_TO_LOCAL=True` to download the JSON from Colab.

In [ ]:
from datetime import datetime, timezone

RESULTS_PATH = ROLE3_ROOT / "reports/role3_real_inference_results.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
all_runs_success = bool(RUN_RESULTS) and all(item["status"] == "success" for item in RUN_RESULTS)
if all_runs_success and TRAINING_CHECK.get("status") == "success":
    overall_status = "completed"
elif any(item["status"] == "success" for item in RUN_RESULTS):
    overall_status = "partial"
else:
    overall_status = "failed"

report = {
    "schema_version": "1.0",
    "status": overall_status,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository": {
        "url": REPO_URL,
        "role3_branch": ROLE3_BRANCH,
        "role3_commit": role3_commit,
        "preprocessing_branch": PREPROCESSING_BRANCH,
        "preprocessing_commit": sarthak_commit,
    },
    "dataset": {
        "repo_id": DATASET_REPO_ID,
        "revision": DATA_REVISION,
        "annotation_file": ANNOTATION_FILENAME,
        "annotation_path_in_runtime": str(ANNOTATION_PATH) if ANNOTATION_PATH is not None else None,
        "validation_record_count": len(VALIDATION_ROWS) if VALIDATION_ROWS else None,
        "sample_ids": SAMPLE_IDS,
        "video_paths": ACTUAL_VIDEO_PATHS,
        "access_error": DATA_SETUP_ERROR,
        "per_frame_count_access_errors": DATA_ACCESS_ERRORS,
    },
    "model": {
        "checkpoint_id": CLIP_MODEL_ID,
        "revision": MODEL_REVISION,
        "class": "CLIPVisionModelWithProjection",
        "projected_dimension": 512,
        "load_error": MODEL_LOAD_ERROR,
    },
    "environment": {
        "python_version": platform.python_version(),
        "pytorch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "device": str(DEVICE),
        "gpu_name": GPU_NAME,
        "gpu_total_memory_mb": GPU_TOTAL_MEMORY_MB,
        "gpu_free_memory_at_start_mb": GPU_FREE_MEMORY_MB,
    },
    "preprocessing": {
        "sampling": "uniform",
        "frame_counts": list(FRAME_COUNTS),
        "batch_size": 1,
        "image_size": [224, 224],
        "channel_order": "RGB (source decoder converts OpenCV BGR to RGB)",
        "input_dtype": "torch.float32",
        "input_normalization": "ImageNet mean [0.485,0.456,0.406], std [0.229,0.224,0.225]; extractor converts to CLIP normalization",
        "geometry": "existing Sarthak square resize; no CLIP center crop",
        "missing_video": "raise",
    },
    "inference_timing_scope": "pipeline forward only; excludes dataset fetch/decode and host-to-device transfer",
    "runs": RUN_RESULTS,
    "training_behavior_check": TRAINING_CHECK,
}
with RESULTS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
    handle.write("\n")
print("Saved report:", RESULTS_PATH)
print("Overall status:", overall_status)

DOWNLOAD_RESULTS_TO_LOCAL = False
if DOWNLOAD_RESULTS_TO_LOCAL:
    from google.colab import files
    files.download(str(RESULTS_PATH))